# Rerank 重排序（Re-ranking）

## 用途
教学演示 - 对检索结果进行精排序提升精度

## 核心概念
- 召回 vs 精排
- Cross-Encoder vs Bi-Encoder
- Rerank 算法（ColBERT、BGE-Reranker）

## 运行前检查

1. 已安装依赖：`pip install -r ../requirements.txt`
2. 已完成 `04_hybrid_search.py` 的学习

In [1]:
from rag_examples.milvus_config import MILVUS_URI
from dotenv import load_dotenv
load_dotenv()

True

# 第一部分：理解 Rerank

## 什么是 Rerank（重排序）？

### 定义
Rerank = 对初步检索的结果重新打分排序，用更精确但更慢的模型，对候选集进行精排序

### 💡 生活化比喻
Rerank = "面试筛选"

**第 1 轮（检索/召回）：**
- 收到 1000 份简历
- 快速筛选，选出 50 个候选人
- 标准：学历、工作年限（快速但粗糙）

**第 2 轮（Rerank/精排）：**
- 50 个候选人参加面试
- 深入评估，选出 Top-5
- 标准：专业能力、沟通能力（慢但精准）

### 📦 为什么要 Rerank？

**检索系统的两难困境：**
- 快：简单模型，但精度低
- 准：复杂模型，但速度慢

**解决方案：两级架构**
```
        用户查询
           ↓
    ┌──────────────┐
    │  检索 (快)     │ → Top-50 候选
    │  Bi-Encoder  │
    └──────┬───────┘
           ↓
    ┌──────────────┐
    │  Rerank(准)   │ → Top-5 最终
    │ Cross-Encoder│
    └──────────────┘
```

### 🔑 Bi-Encoder vs Cross-Encoder

**Bi-Encoder（检索用）:**
- 分别编码查询和文档
- `query → [0.1, 0.2, ...]`
- `doc1  → [0.3, 0.4, ...]`
- 计算余弦相似度
- 优点：快，可预先计算文档向量
- 缺点：query 和 doc 无交互，精度有限

**Cross-Encoder（Rerank 用）:**
- 拼接 query 和 doc 一起编码
- `[query, doc] → 相关性分数`
- 优点：充分交互，精度高
- 缺点：慢，无法预先计算

### ⚠️ 常见 Rerank 模型

| 模型 | 说明 |
|------|------|
| BGE-Reranker（智源） | 中文效果好，推荐 bge-reranker-large |
| ColBERT | 细粒度交互，精度高 |
| Cross-Encoder (sentence-transformers) | ms-marco-MiniLM-L-12-v2 |
| Cohere Rerank API | 云端 API，效果好但有成本 |

In [2]:
def mock_rerank_pipeline():
    """
    模拟 Rerank 流程演示
    """
    print("=" * 60)
    print("示例 1: 模拟 Rerank 流程")
    print("=" * 60)

    # 用户查询
    query = "机器学习需要什么基础？"

    # 初步检索结果（假设从向量检索获得）
    retrieved_docs = [
        (0.85, "深度学习是机器学习的子集，使用神经网络。"),  # 相似度 0.85
        (0.82, "机器学习通过训练数据让计算机自动学习。"),     # 相似度 0.82
        (0.78, "Python 是常用的编程语言，用于 AI 开发。"),    # 相似度 0.78
        (0.75, "数据结构和算法是编程的基础。"),              # 相似度 0.75
        (0.72, "机器学习需要数学基础，包括线性代数和概率。"),  # 相似度 0.72
    ]

    print(f"用户查询：{query}")
    print()

    print("1. 初步检索结果（按向量相似度排序）：")
    print("-" * 50)
    for i, (score, doc) in enumerate(retrieved_docs):
        print(f"  [{i+1}] 相似度：{score:.2f}")
        print(f"      {doc}")
    print()

    # 模拟 Rerank 分数（假设 Rerank 模型更懂"基础"这个词）
    rerank_scores = [
        0.45,  # 第 1 句：讲深度学习，不是基础
        0.68,  # 第 2 句：讲机器学习是什么
        0.32,  # 第 3 句：讲 Python，偏离主题
        0.28,  # 第 4 句：讲编程基础，不是机器学习
        0.92,  # 第 5 句：直接回答"需要数学基础"
    ]

    # 重新排序
    reranked = list(zip(retrieved_docs, rerank_scores))
    reranked.sort(key=lambda x: x[1], reverse=True)

    print("2. Rerank 后结果（按相关性重新排序）：")
    print("-" * 50)
    for i, ((orig_score, doc), rerank_score) in enumerate(reranked):
        orig_idx = next(j for j, (s, d) in enumerate(retrieved_docs) if s == orig_score and d == doc)
        rank_change = "↑" if i < orig_idx else "↓" if i > orig_idx else "="
        print(f"  [{i+1}] Rerank 分数：{rerank_score:.2f} {rank_change}")
        print(f"      原相似度：{orig_score:.2f}")
        print(f"      {doc}")
    print()

    print("💡 观察：")
    print("   - 第 5 句从第 5 名上升到第 1 名（直接回答问题）")
    print("   - 第 1 句从第 1 名下降到第 3 名（讲的是子集概念）")
    print("   - 这就是 Rerank 的价值：更精准理解 query-doc 相关性")

In [3]:
mock_rerank_pipeline()

示例 1: 模拟 Rerank 流程
用户查询：机器学习需要什么基础？

1. 初步检索结果（按向量相似度排序）：
--------------------------------------------------
  [1] 相似度：0.85
      深度学习是机器学习的子集，使用神经网络。
  [2] 相似度：0.82
      机器学习通过训练数据让计算机自动学习。
  [3] 相似度：0.78
      Python 是常用的编程语言，用于 AI 开发。
  [4] 相似度：0.75
      数据结构和算法是编程的基础。
  [5] 相似度：0.72
      机器学习需要数学基础，包括线性代数和概率。

2. Rerank 后结果（按相关性重新排序）：
--------------------------------------------------
  [1] Rerank 分数：0.92 ↑
      原相似度：0.72
      机器学习需要数学基础，包括线性代数和概率。
  [2] Rerank 分数：0.68 =
      原相似度：0.82
      机器学习通过训练数据让计算机自动学习。
  [3] Rerank 分数：0.45 ↓
      原相似度：0.85
      深度学习是机器学习的子集，使用神经网络。
  [4] Rerank 分数：0.32 ↓
      原相似度：0.78
      Python 是常用的编程语言，用于 AI 开发。
  [5] Rerank 分数：0.28 ↓
      原相似度：0.75
      数据结构和算法是编程的基础。

💡 观察：
   - 第 5 句从第 5 名上升到第 1 名（直接回答问题）
   - 第 1 句从第 1 名下降到第 3 名（讲的是子集概念）
   - 这就是 Rerank 的价值：更精准理解 query-doc 相关性


# 第二部分：使用 BGE-Reranker

In [6]:
def bge_reranker_demo():
    """
    使用 BGE-Reranker 进行重排序
    """
    print("=" * 60)
    print("示例 2: 使用 BGE-Reranker")
    print("=" * 60)

    try:
        from FlagEmbedding import FlagReranker

        # 查询和候选文档
        query = "机器学习需要什么基础？"

        candidates = [
            "深度学习是机器学习的子集，使用神经网络。",
            "机器学习通过训练数据让计算机自动学习。",
            "Python 是常用的编程语言，用于 AI 开发。",
            "数据结构和算法是编程的基础。",
            "机器学习需要数学基础，包括线性代数和概率统计。",
        ]

        print(f"查询：{query}")
        print(f"候选文档：{len(candidates)} 条\n")

        # 初始化 Reranker
        print("加载 BGE-Reranker 模型...")
        reranker = FlagReranker('BAAI/bge-reranker-large', use_fp16=False)
        print("✓ 模型加载完成\n")

        # 计算分数
        print("计算相关性分数...")
        scores = reranker.compute_score([query, doc] for doc in candidates)

        # 排序
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

        print("\nRerank 结果：")
        print("-" * 50)
        for rank, (idx, score) in enumerate(ranked, 1):
            print(f"  [{rank}] 分数：{score:.4f}")
            print(f"      {candidates[idx]}")
            print()

    except ImportError:
        print("需要安装：pip install FlagEmbedding")
        print("\n模拟演示（无真实模型）：")
        print("-" * 50)

        query = "机器学习需要什么基础？"
        candidates = [
            "深度学习是机器学习的子集，使用神经网络。",
            "机器学习通过训练数据让计算机自动学习。",
            "Python 是常用的编程语言，用于 AI 开发。",
            "数据结构和算法是编程的基础。",
            "机器学习需要数学基础，包括线性代数和概率统计。",
        ]

        # 模拟分数
        mock_scores = [0.45, 0.68, 0.32, 0.28, 0.92]

        ranked = sorted(zip(range(len(candidates)), mock_scores), key=lambda x: x[1], reverse=True)

        for rank, (idx, score) in enumerate(ranked, 1):
            print(f"  [{rank}] 分数：{score:.2f}")
            print(f"      {candidates[idx]}")

In [5]:
bge_reranker_demo()

示例 2: 使用 BGE-Reranker
需要安装：pip install FlagEmbedding

模拟演示（无真实模型）：
--------------------------------------------------
  [1] 分数：0.92
      机器学习需要数学基础，包括线性代数和概率统计。
  [2] 分数：0.68
      机器学习通过训练数据让计算机自动学习。
  [3] 分数：0.45
      深度学习是机器学习的子集，使用神经网络。
  [4] 分数：0.32
      Python 是常用的编程语言，用于 AI 开发。
  [5] 分数：0.28
      数据结构和算法是编程的基础。


# 第三部分：Cross-Encoder Rerank

In [7]:
def cross_encoder_rerank():
    """
    使用 sentence-transformers 的 CrossEncoder
    """
    print()
    print("=" * 60)
    print("示例 3: CrossEncoder Rerank")
    print("=" * 60)

    try:
        from sentence_transformers import CrossEncoder

        query = "如何学习人工智能？"

        candidates = [
            "人工智能是计算机科学的一个分支。",
            "学习 AI 需要掌握 Python 编程和数学基础。",
            "深度学习使用神经网络处理复杂任务。",
            "推荐系统根据用户行为推荐内容。",
            "人工智能包括机器学习和知识工程。",
        ]

        print(f"查询：{query}")
        print(f"候选文档：{len(candidates)} 条\n")

        # 加载 CrossEncoder 模型
        print("加载 CrossEncoder 模型...")
        model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')
        print("✓ 模型加载完成\n")

        # 计算分数
        print("计算相关性分数...")
        pairs = [[query, doc] for doc in candidates]
        scores = model.predict(pairs)

        # 排序
        ranked = sorted(zip(range(len(candidates)), scores), key=lambda x: x[1], reverse=True)

        print("\nRerank 结果：")
        print("-" * 50)
        for rank, (idx, score) in enumerate(ranked, 1):
            print(f"  [{rank}] 分数：{score:.4f}")
            print(f"      {candidates[idx]}")
        print()

    except ImportError:
        print("需要安装：pip install sentence-transformers")
        print("\n模拟演示：")
        print("-" * 50)

        query = "如何学习人工智能？"
        candidates = [
            "人工智能是计算机科学的一个分支。",
            "学习 AI 需要掌握 Python 编程和数学基础。",
            "深度学习使用神经网络处理复杂任务。",
            "推荐系统根据用户行为推荐内容。",
            "人工智能包括机器学习和知识工程。",
        ]

        # 模拟分数
        mock_scores = [0.35, 0.89, 0.52, 0.28, 0.41]
        ranked = sorted(zip(range(len(candidates)), mock_scores), key=lambda x: x[1], reverse=True)

        for rank, (idx, score) in enumerate(ranked, 1):
            print(f"  [{rank}] 分数：{score:.2f}")
            print(f"      {candidates[idx]}")

In [8]:
cross_encoder_rerank()


示例 3: CrossEncoder Rerank
需要安装：pip install sentence-transformers

模拟演示：
--------------------------------------------------
  [1] 分数：0.89
      学习 AI 需要掌握 Python 编程和数学基础。
  [2] 分数：0.52
      深度学习使用神经网络处理复杂任务。
  [3] 分数：0.41
      人工智能包括机器学习和知识工程。
  [4] 分数：0.35
      人工智能是计算机科学的一个分支。
  [5] 分数：0.28
      推荐系统根据用户行为推荐内容。


# 第四部分：Rerank 性能对比

In [9]:
def rerank_performance_comparison():
    """
    对比 Rerank 前后的效果
    """
    print()
    print("=" * 60)
    print("示例 4: Rerank 性能对比")
    print("=" * 60)

    # 测试查询
    query = "RAG 系统如何工作？"

    # 候选文档（按向量检索的原始排名）
    candidates = [
        ("向量检索", 0.88, "向量检索是根据查询向量查找相似文档的过程。"),
        ("RAG 简介", 0.85, "RAG 是检索增强生成，结合检索和生成的 AI 技术。"),
        ("Embedding", 0.79, "Embedding 将文本转换为数字向量。"),
        ("生成模型", 0.76, "生成模型根据输入生成新的内容，如 GPT。"),
        ("RAG 流程", 0.73, "RAG 的工作流程：检索相关文档，将文档和问题一起交给 LLM 生成答案。"),
        ("Milvus", 0.70, "Milvus 是向量数据库，用于存储和检索向量。"),
    ]

    print(f"查询：{query}")
    print()

    # 原始排名
    print("1. 向量检索结果（原始排名）：")
    print("-" * 50)
    for i, (name, score, _) in enumerate(candidates):
        print(f"  [{i+1}] {score:.2f} - {name}")
    print()

    # 模拟 Rerank 分数
    rerank_scores = [0.42, 0.68, 0.35, 0.48, 0.91, 0.29]

    # 重新排序
    reranked = list(zip(candidates, rerank_scores))
    reranked.sort(key=lambda x: x[1], reverse=True)

    print("2. Rerank 后结果：")
    print("-" * 50)
    for i, ((name, orig_score, content), rerank_score) in enumerate(reranked):
        orig_rank = next(j + 1 for j, (n, s, _) in enumerate(candidates) if n == name)
        new_rank = i + 1
        change = new_rank - orig_rank
        arrow = "↑" if change < 0 else "↓" if change > 0 else "="
        print(f"  [{new_rank}] {rerank_score:.2f} ({orig_rank}{arrow}) - {name}")
        if i < 3:
            print(f"      {content}")
    print()

    print("3. 效果对比：")
    print("-" * 50)

    # 原始排名的相关性（0-3 分）
    orig_relevance = [1, 2, 0, 1, 3, 0]

    # Rerank 后的相关性
    reranked_relevance = [3, 2, 1, 1, 0, 0]

    # 计算 DCG
    def dcg(relevances):
        return sum(rel / (i + 1) for i, rel in enumerate(relevances))

    orig_dcg = dcg([1, 2, 0, 1, 3, 0])
    reranked_dcg = dcg([3, 2, 1, 1, 0, 0])
    ideal_dcg = dcg([3, 2, 1, 1, 0, 0])

    print(f"  原始 DCG: {orig_dcg:.2f}")
    print(f"  Rerank DCG: {reranked_dcg:.2f}")
    print(f"  理想 DCG: {ideal_dcg:.2f}")
    print(f"  NDCG 提升：{(reranked_dcg - orig_dcg) / ideal_dcg * 100:.1f}%")

In [10]:
rerank_performance_comparison()


示例 4: Rerank 性能对比
查询：RAG 系统如何工作？

1. 向量检索结果（原始排名）：
--------------------------------------------------
  [1] 0.88 - 向量检索
  [2] 0.85 - RAG 简介
  [3] 0.79 - Embedding
  [4] 0.76 - 生成模型
  [5] 0.73 - RAG 流程
  [6] 0.70 - Milvus

2. Rerank 后结果：
--------------------------------------------------
  [1] 0.91 (5↑) - RAG 流程
      RAG 的工作流程：检索相关文档，将文档和问题一起交给 LLM 生成答案。
  [2] 0.68 (2=) - RAG 简介
      RAG 是检索增强生成，结合检索和生成的 AI 技术。
  [3] 0.48 (4↑) - 生成模型
      生成模型根据输入生成新的内容，如 GPT。
  [4] 0.42 (1↓) - 向量检索
  [5] 0.35 (3↓) - Embedding
  [6] 0.29 (6=) - Milvus

3. 效果对比：
--------------------------------------------------
  原始 DCG: 2.85
  Rerank DCG: 4.58
  理想 DCG: 4.58
  NDCG 提升：37.8%


# 第五部分：完整的 RAG 检索 + Rerank 流程

In [ ]:
def complete_rag_rerank_pipeline():
    """
    完整的 RAG 检索 + Rerank 流程
    """
    print()
    print("=" * 60)
    print("示例 5: 完整 RAG + Rerank 流程")
    print("=" * 60)

    print("""
完整的 RAG 检索流程（带 Rerank）:

┌─────────────────────────────────────────────────────────┐
│                                                         │
│  1. 用户提问                                             │
│     → "机器学习和深度学习有什么区别？"                   │
│                                                         │
│  2. 生成查询向量                                        │
│     → Embedding 模型编码                                 │
│                                                         │
│  3. 向量检索（召回）                                    │
│     → 从 Milvus 检索 Top-50 候选文档                     │
│                                                         │
│  4. Rerank 重排序（精排）                               │
│     → 用 CrossEncoder 对 50 个候选重新打分               │
│     → 选取 Top-5 最终结果                               │
│                                                         │
│  5. 构建上下文                                          │
│     → 拼接 Top-5 文档内容                                │
│                                                         │
│  6. 调用 LLM 生成答案                                    │
│     → Prompt: 根据以下信息回答问题...                    │
│                                                         │
└─────────────────────────────────────────────────────────┘

💡 代码示例:

```python
def rag_with_rerank(query, top_k=5):
    # 1. 生成查询向量
    query_vector = embedding_model.encode(query)

    # 2. 向量检索（召回 Top-50）
    results = milvus.search(
        collection_name="knowledge_base",
        data=[query_vector],
        limit=50
    )

    # 3. 提取候选文档
    candidates = [hit['content'] for hit in results[0]]

    # 4. Rerank 重排序
    pairs = [[query, doc] for doc in candidates]
    scores = reranker.predict(pairs)

    # 5. 选取 Top-K
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    top_docs = [doc for doc, _ in ranked[:top_k]]

    # 6. 构建上下文
    context = "\\n\\n".join(top_docs)

    # 7. 调用 LLM
    prompt = f"根据以下信息回答问题：\\n\\n{context}\\n\\n问题：{query}"
    answer = llm.generate(prompt)

    return answer, top_docs
```
""")

In [ ]:
complete_rag_rerank_pipeline()

# 第六部分：Rerank 最佳实践

In [ ]:
def rerank_best_practices():
    """
    Rerank 最佳实践
    """
    print()
    print("=" * 60)
    print("示例 6: Rerank 最佳实践")
    print("=" * 60)

    print("""
┌─────────────────────────────────────────────────────────┐
│ Rerank 最佳实践                                         │
├─────────────────────────────────────────────────────────┤
│                                                         │
│ 1. 何时使用 Rerank                                      │
│    ✓ 对检索精度要求高的场景                              │
│    ✓ 候选集较小（<100 条）                              │
│    ✓ 有足够的计算资源                                   │
│    ✗ 实时性要求极高（Rerank 会增加延迟）                │
│    ✗ 候选集太大（成本过高）                            │
│                                                         │
│ 2. 模型选择                                             │
│    中文场景：BAAI/bge-reranker-large                    │
│    英文场景：cross-encoder/ms-marco-MiniLM-L-12-v2     │
│    多语言：BAAI/bge-reranker-v2-m3                      │
│    API 方案：Cohere Rerank API                          │
│                                                         │
│ 3. 参数建议                                             │
│    - 召回数量：50-100 条（给 Rerank 足够选择）           │
│    - 返回数量：5-10 条（LLM 上下文限制）                │
│    - 分数阈值：过滤掉<0.3 的低相关结果                  │
│                                                         │
│ 4. 性能优化                                             │
│    - 使用 fp16 推理（节省显存）                         │
│    - 批量处理：一次预测多对 query-doc                   │
│    - 缓存热点查询的 Rerank 结果                          │
│                                                         │
│ 5. 成本控制                                             │
│    - 只在必要时使用（高价值查询）                       │
│    - 先用轻量模型粗排，再用重量模型精排                │
│    - 考虑蒸馏：用大模型训练小模型                       │
│                                                         │
└─────────────────────────────────────────────────────────┘

📊 典型延迟对比:

| 阶段 | 模型 | 延迟 (单次) |
|------|------|-------------|
| 检索 | Bi-Encoder | ~10ms |
| Rerank | CrossEncoder | ~100-500ms |
| 生成 | LLM | ~1-5s |

💡 建议：Rerank 增加的延迟（100-500ms）通常值得，
因为检索质量提升可以显著改善最终答案质量。
""")

In [ ]:
rerank_best_practices()

# 学习总结

## 关键概念

| 概念 | 说明 |
|------|------|
| Rerank | 对检索结果重新打分排序 |
| Bi-Encoder | 分别编码 query 和 doc，快但精度有限 |
| Cross-Encoder | 拼接编码，充分交互，精度高 |
| NDCG | 衡量排序质量的指标 |

## 下一步学习

- RAG API 封装